# DHd Blog Web Scraper

## 1. Environment Setup and Shared Utilities


In [57]:
# ============================================================
# SECTION 1: ENVIRONMENT SETUP AND SHARED UTILITIES
# ============================================================
# Purpose:
# - Import libraries used throughout the scraper.
# - Configure project paths and website access.
# - Define reusable functions for HTML retrieval, text cleaning,
#   and post identification.
# ============================================================


# ------------------------------------------------------------
# 1.1 IMPORT LIBRARIES
# ------------------------------------------------------------

# Standard library
import re
import time
from pathlib import Path
from urllib.parse import parse_qs, urljoin, urlparse

# Third-party libraries
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1.2 CONFIGURE PROJECT PATHS
# ------------------------------------------------------------
# Resolve the project directory automatically, whether the
# notebook runs from the repository root or notebooks folder.

try:
    CURRENT_DIR = Path(__file__).resolve().parent
except NameError:
    CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name.lower() == "notebooks"
    else CURRENT_DIR
)

# Create the data directory if necessary.
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1.3 CONFIGURE WEBSITE ACCESS
# ------------------------------------------------------------
# Reuse one HTTP session for requests to the DHd Blog.

BASE_URL = "https://dhd-blog.org/"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36 "
        "Academic DHd Blog analysis"
    )
}

session = requests.Session()
session.headers.update(HEADERS)


# ------------------------------------------------------------
# 1.4 DEFINE SHARED CONSTANTS
# ------------------------------------------------------------

# German month names used to recognize archive links.
GERMAN_MONTHS = (
    "Januar|Februar|März|April|Mai|Juni|Juli|"
    "August|September|Oktober|November|Dezember"
)


# ------------------------------------------------------------
# 1.5 DEFINE WEB REQUEST HELPER
# ------------------------------------------------------------

def get_soup(url, retries=4, timeout=40):
    """
    Retrieve a webpage and parse its HTML.

    Retry failed requests using exponential backoff.
    """

    last_error = None

    for attempt in range(retries):
        try:
            response = session.get(url, timeout=timeout)
            response.raise_for_status()

            return BeautifulSoup(response.content, "lxml")

        except requests.RequestException as error:
            last_error = error

            if attempt < retries - 1:
                time.sleep(2 ** attempt)

    raise last_error


# ------------------------------------------------------------
# 1.6 DEFINE TEXT CLEANING HELPER
# ------------------------------------------------------------

def clean_spacing(text):
    """
    Normalize whitespace while preserving paragraph breaks.
    """

    if not text:
        return ""

    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# ------------------------------------------------------------
# 1.7 DEFINE POST IDENTIFICATION HELPER
# ------------------------------------------------------------

def get_post_id(url):
    """
    Extract the numeric WordPress post ID from a URL.

    Works with both original WordPress and static HTML URLs.
    """

    query = parse_qs(urlparse(url).query)
    post_id = query.get("p", [""])[0]

    return str(post_id) if str(post_id).isdigit() else ""


# ------------------------------------------------------------
# 1.8 CONFIRM INITIALIZATION
# ------------------------------------------------------------

print("SECTION 1 COMPLETED")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Website: {BASE_URL}")


SECTION 1 COMPLETED
Project root: C:\Users\jiwani\Documents\dhd-blog-analysis
Data directory: C:\Users\jiwani\Documents\dhd-blog-analysis\data
Website: https://dhd-blog.org/


## 2. Archive Discovery and Post Link Collection


In [58]:
# ============================================================
# SECTION 2: ARCHIVE DISCOVERY AND POST LINK COLLECTION
# ============================================================
# Purpose:
# - Discover monthly archives from the DHd Blog homepage.
# - Traverse archive pagination to collect all post links.
# - Preserve actual static HTML article URLs.
# - Deduplicate posts using their unique WordPress IDs.
# - Compare discovered posts with the existing dataset.
#
# Note:
# The DHd Blog currently uses static HTML URLs. These URLs
# must be preserved rather than converted to WordPress URLs.
# ============================================================


# ------------------------------------------------------------
# 2.1 DISCOVER MONTHLY ARCHIVES
# ------------------------------------------------------------

def get_archive_months():
    """
    Extract monthly archive links from the DHd Blog homepage.
    """

    soup = get_soup(BASE_URL)
    months = {}

    for link in soup.find_all("a", href=True):
        label = clean_spacing(link.get_text(" ", strip=True))

        # Identify archive links labeled with a German month and year.
        if re.fullmatch(rf"(?:{GERMAN_MONTHS})\s+\d{{4}}", label):
            archive_url = urljoin(BASE_URL, link["href"])
            months[label] = archive_url

    return pd.DataFrame(
        [
            {"archive_month": month, "archive_url": url}
            for month, url in months.items()
        ]
    )


# ------------------------------------------------------------
# 2.2 EXTRACT POST LINKS FROM AN ARCHIVE PAGE
# ------------------------------------------------------------

def extract_post_links_from_archive(soup, archive_url):
    """
    Extract unique article links while preserving their
    original static HTML filenames.

    Example:
    https://dhd-blog.org/index5caa.html?p=24377
    """

    post_urls = set()

    # Support different HTML structures used by the blog.
    selectors = [
        "h2 a[rel='bookmark'][href]",
        "#content h2 a[href]",
        "article h2 a[href]",
        "article h1 a[href]"
    ]

    for selector in selectors:
        for link in soup.select(selector):
            full_url = urljoin(archive_url, link["href"])

            if get_post_id(full_url):
                post_urls.add(full_url)

    # Fallback for older archive layouts.
    if not post_urls:
        for heading in soup.find_all(["h1", "h2"]):
            link = heading.find("a", href=True)

            if not link:
                continue

            full_url = urljoin(archive_url, link["href"])

            if get_post_id(full_url):
                post_urls.add(full_url)

    return post_urls


# ------------------------------------------------------------
# 2.3 IDENTIFY THE NEXT ARCHIVE PAGE
# ------------------------------------------------------------

def get_next_archive_page(soup, current_url, current_page):
    """
    Find the next archive page using actual pagination links.

    Static archive pages may use different HTML filenames.
    """

    current_query = parse_qs(urlparse(current_url).query)
    current_month = current_query.get("m", [None])[0]

    candidates = []

    for link in soup.find_all("a", href=True):
        next_url = urljoin(current_url, link["href"])
        query = parse_qs(urlparse(next_url).query)

        # Only follow pagination within the same archive month.
        if query.get("m", [None])[0] != current_month:
            continue

        page_values = query.get("paged", [])

        if not page_values:
            continue

        try:
            page_number = int(page_values[0])
        except (ValueError, TypeError):
            continue

        if page_number == current_page + 1:
            candidates.append(next_url)

    return candidates[0] if candidates else None


# ------------------------------------------------------------
# 2.4 COLLECT POSTS FROM ALL ARCHIVE PAGES
# ------------------------------------------------------------

archive_df = get_archive_months()

print("Archive months found:", len(archive_df))

post_records = []
archive_errors = []

global_seen_post_ids = set()
visited_archive_pages = set()

# Safety limit to prevent unintended pagination loops.
MAX_PAGES_PER_MONTH = 100

for _, archive in tqdm(
    archive_df.iterrows(),
    total=len(archive_df),
    desc="Collecting all archive pages"
):
    archive_month = archive["archive_month"]
    page_url = archive["archive_url"]

    month_seen_ids = set()
    page_number = 1

    while page_url and page_number <= MAX_PAGES_PER_MONTH:

        # Prevent visiting the same archive URL repeatedly.
        if page_url in visited_archive_pages:
            print(f"Repeated archive URL detected: {page_url}")
            break

        visited_archive_pages.add(page_url)

        try:
            soup = get_soup(page_url)

            page_post_urls = extract_post_links_from_archive(
                soup,
                page_url
            )

        except Exception as error:
            archive_errors.append(
                {
                    "archive_month": archive_month,
                    "archive_page": page_number,
                    "archive_page_url": page_url,
                    "error": str(error)
                }
            )

            print(
                f"Error: {archive_month}, "
                f"page {page_number}: {error}"
            )
            break

        # Extract post IDs for duplicate detection.
        page_post_ids = {
            get_post_id(url)
            for url in page_post_urls
            if get_post_id(url)
        }

        new_month_post_ids = page_post_ids - month_seen_ids

        if not page_post_ids:
            print(
                f"No posts found: {archive_month}, "
                f"page {page_number}"
            )
            break

        if not new_month_post_ids:
            print(
                f"Repeated posts: {archive_month}, "
                f"page {page_number}"
            )
            break

        # Record each unique post and its archive metadata.
        for post_url in sorted(page_post_urls):
            post_id = get_post_id(post_url)

            if not post_id:
                continue

            month_seen_ids.add(post_id)

            if post_id not in global_seen_post_ids:
                global_seen_post_ids.add(post_id)

                post_records.append(
                    {
                        "post_id": post_id,
                        "url": post_url,
                        "archive_month": archive_month,
                        "archive_page": page_number,
                        "archive_page_url": page_url
                    }
                )

        # Follow the next archive page, if one exists.
        next_page_url = get_next_archive_page(
            soup,
            page_url,
            page_number
        )

        if not next_page_url:
            break

        page_url = next_page_url
        page_number += 1

        # Short delay to avoid excessive requests.
        time.sleep(0.4)


# ------------------------------------------------------------
# 2.5 CREATE POST LINK AND ERROR DATAFRAMES
# ------------------------------------------------------------

post_links_df = pd.DataFrame(
    post_records,
    columns=[
        "post_id",
        "url",
        "archive_month",
        "archive_page",
        "archive_page_url"
    ]
)

archive_errors_df = pd.DataFrame(
    archive_errors,
    columns=[
        "archive_month",
        "archive_page",
        "archive_page_url",
        "error"
    ]
)

if post_links_df.empty:
    raise RuntimeError(
        "No posts were found. Check archive HTML."
    )


# ------------------------------------------------------------
# 2.6 COMPARE WITH THE EXISTING DATASET
# ------------------------------------------------------------
# Identify newly discovered posts and check whether any
# previously collected post IDs are missing.

dataset_path = DATA_DIR / "dhd_blog_dataset.csv"

if dataset_path.exists():

    old_df = pd.read_csv(
        dataset_path,
        dtype={"post_id": str}
    )

    old_ids = set(old_df["post_id"].dropna())
    new_ids = set(post_links_df["post_id"].astype(str))

    missing_ids = old_ids - new_ids
    added_ids = new_ids - old_ids

    print("\n========== DATASET COMPARISON ==========")
    print("Previously collected posts:", len(old_ids))
    print("Currently discovered posts:", len(new_ids))
    print("Old posts missing:", len(missing_ids))
    print("Newly discovered posts:", len(added_ids))

    print("\nMissing IDs:", sorted(missing_ids))
    print("\nNew IDs:", sorted(added_ids))

else:
    print("\nNo existing dataset found.")
    print("Dataset comparison skipped.")


print("\nSECTION 2 COMPLETED")
print("Existing dataset files have not been modified.")


Archive months found: 179



========== DATASET COMPARISON ==========
Previously collected posts: 2779
Currently discovered posts: 2780
Old posts missing: 0
Newly discovered posts: 1

Missing IDs: []

New IDs: ['24392']

SECTION 2 COMPLETED
Existing dataset files have not been modified.


In [59]:
# ------------------------------------------------------------
# 2.7 VALIDATE ARCHIVE COVERAGE AND COLLECTED POST LINKS
# ------------------------------------------------------------
# Check archive pagination, unique post IDs, collection
# completeness, and any errors encountered during discovery.

print("\n========== ARCHIVE COLLECTION SUMMARY ==========")

# Archive coverage
print("Archive months discovered:", len(archive_df))
print("Highest archive page:", post_links_df["archive_page"].max())

print("\nPosts collected by archive page:")
display(
    post_links_df["archive_page"]
    .value_counts()
    .sort_index()
    .rename_axis("archive_page")
    .reset_index(name="posts")
)

# Post link validation
print("\n========== POST LINK VALIDATION ==========")
print("Total collected posts:", len(post_links_df))
print("Unique post IDs:", post_links_df["post_id"].nunique())
print("Duplicate post IDs:", post_links_df["post_id"].duplicated().sum())
print("Archive errors:", len(archive_errors_df))

# Verify that no duplicate post IDs were collected.
assert post_links_df["post_id"].is_unique, (
    "Duplicate post IDs detected!"
)

# Preview collected static article URLs.
print("\nSample collected post links:")
display(
    post_links_df[
        ["post_id", "url", "archive_month", "archive_page"]
    ].head(10)
)

print("\nSection 2.6 validation completed.")



========== ARCHIVE COLLECTION SUMMARY ==========
Archive months discovered: 179
Highest archive page: 5

Posts collected by archive page:


,archive_page,posts
0,1,1733
1,2,900
2,3,123
3,4,23
4,5,1



========== POST LINK VALIDATION ==========
Total collected posts: 2780
Unique post IDs: 2780
Duplicate post IDs: 0
Archive errors: 0

Sample collected post links:


,post_id,url,archive_month,archive_page
0,24360,https://dhd-blog.org/index0220.html?p=24360,Oktober 2026,1
1,24331,https://dhd-blog.org/index1854.html?p=24331,Oktober 2026,1
2,24319,https://dhd-blog.org/index4b70.html?p=24319,Oktober 2026,1
3,24377,https://dhd-blog.org/index5caa.html?p=24377,Oktober 2026,1
4,24392,https://dhd-blog.org/index5f66.html?p=24392,Oktober 2026,1
5,24285,https://dhd-blog.org/index7845.html?p=24285,Oktober 2026,1
6,24314,https://dhd-blog.org/index7a33.html?p=24314,Oktober 2026,1
7,24358,https://dhd-blog.org/indexc8ea.html?p=24358,Oktober 2026,1
8,24305,https://dhd-blog.org/indexe4f7.html?p=24305,Oktober 2026,1
9,24309,https://dhd-blog.org/indexe5d3.html?p=24309,Oktober 2026,1



Section 2.6 validation completed.


## 3. Article Content Extraction, Checkpointed Scraping, and Data Validation


#### 3.1	Article Extraction Functions

In [60]:
# ============================================================
# SECTION 3: ARTICLE EXTRACTION, SCRAPING, AND VALIDATION
# PART 1: ARTICLE EXTRACTION FUNCTIONS
# ============================================================
# Purpose:
# - Extract article metadata and full text from static URLs.
# - Clean article text and calculate word counts.
# - Detect temporary website announcements.
# - Define the function used to scrape individual posts.
#
# Requires the imports and shared utilities from Section 1.
# ============================================================


# ------------------------------------------------------------
# 3.1.1 DEFINE GENERAL EXTRACTION HELPERS
# ------------------------------------------------------------

def safe_url_info(href):
    """Resolve a relative URL and return its parsed components."""
    if not href:
        return None, None

    try:
        full_url = urljoin(BASE_URL, href)
        return full_url, urlparse(full_url)
    except (ValueError, TypeError):
        return None, None


def unique_clean_values(values):
    """Remove empty and duplicate values while preserving order."""
    result = []
    seen = set()

    for value in values:
        value = clean_spacing(str(value))

        if value and value not in seen:
            seen.add(value)
            result.append(value)

    return result


def first_matching_text(container, selectors):
    """Extract text from the first matching HTML element."""
    for selector in selectors:
        element = container.select_one(selector)

        if element:
            value = clean_spacing(
                element.get_text(" ", strip=True)
            )

            if value:
                return value

    return ""


# ------------------------------------------------------------
# 3.1.2 LOCATE THE MAIN ARTICLE
# ------------------------------------------------------------

def find_article(soup):
    """Identify the main article container in the webpage."""

    selectors = [
        "#content article.article",
        "#content article",
        "div#content article",
        "article.article",
        "article",
        ".post"
    ]

    for selector in selectors:
        article = soup.select_one(selector)

        if article is not None:
            return article

    return None


# ------------------------------------------------------------
# 3.1.3 EXTRACT ARTICLE TITLE AND AUTHOR
# ------------------------------------------------------------

def extract_title(article):
    """Extract the article title."""

    return first_matching_text(
        article,
        [
            "h1.entry-title",
            "h2.entry-title",
            "h1 a[rel='bookmark']",
            "h2 a[rel='bookmark']",
            "h1",
            "h2"
        ]
    )


def extract_author(article):
    """Extract the author from metadata or publication information."""

    author = first_matching_text(
        article,
        [
            "a[rel='author']",
            ".author a",
            ".byline a",
            ".vcard a"
        ]
    )

    if author:
        return author

    # Fallback for German publication metadata.
    text = article.get_text(" ", strip=True)

    match = re.search(
        r"Veröffentlicht\s+von\s+(.+?)\s+am\s+"
        r"\d{1,2}\.\s+\w+\s+\d{4}",
        text,
        flags=re.IGNORECASE
    )

    return clean_spacing(match.group(1)) if match else ""


# ------------------------------------------------------------
# 3.1.4 EXTRACT PUBLICATION DATE
# ------------------------------------------------------------

MONTH_NUMBERS = {
    "januar": 1,
    "februar": 2,
    "märz": 3,
    "maerz": 3,
    "april": 4,
    "mai": 5,
    "juni": 6,
    "juli": 7,
    "august": 8,
    "september": 9,
    "oktober": 10,
    "november": 11,
    "dezember": 12
}


def extract_publication_date(article):
    """Extract the publication date in YYYY-MM-DD format."""

    # Prefer machine-readable dates when available.
    element = article.select_one("time[datetime]")

    if element:
        raw_date = element.get("datetime", "")
        match = re.search(r"\d{4}-\d{2}-\d{2}", raw_date)

        if match:
            return match.group(0)

    # Fallback for German dates displayed in article metadata.
    text = article.get_text(" ", strip=True)

    match = re.search(
        r"(\d{1,2})\.\s*"
        r"(Januar|Februar|März|Maerz|April|Mai|Juni|"
        r"Juli|August|September|Oktober|November|Dezember)"
        r"\s+(\d{4})",
        text,
        flags=re.IGNORECASE
    )

    if not match:
        return ""

    day = int(match.group(1))
    month = MONTH_NUMBERS.get(match.group(2).lower())
    year = int(match.group(3))

    if month is None:
        return ""

    try:
        return pd.Timestamp(
            year=year,
            month=month,
            day=day
        ).strftime("%Y-%m-%d")
    except ValueError:
        return ""


# ------------------------------------------------------------
# 3.1.5 EXTRACT ARTICLE CATEGORIES
# ------------------------------------------------------------

def extract_categories(article):
    """Extract WordPress categories from article metadata."""

    categories = []

    for link in article.find_all("a", href=True):
        _, parsed = safe_url_info(link["href"])

        if parsed is None:
            continue

        rel_values = link.get("rel", [])

        if isinstance(rel_values, str):
            rel_values = rel_values.split()

        rel_values = [value.lower() for value in rel_values]

        is_category = (
            "/category/" in parsed.path.lower()
            or "category" in rel_values
        )

        if is_category:
            categories.append(
                link.get_text(" ", strip=True)
            )

    # Additional selectors for alternative WordPress layouts.
    for link in article.select(
        ".cat-links a, "
        ".post-categories a, "
        ".entry-categories a"
    ):
        categories.append(
            link.get_text(" ", strip=True)
        )

    return unique_clean_values(categories)


# ------------------------------------------------------------
# 3.1.6 EXTRACT ARTICLE TAGS
# ------------------------------------------------------------

def extract_tags(article):
    """Extract tags while excluding category metadata."""

    tags = []

    for link in article.find_all("a", href=True):
        _, parsed = safe_url_info(link["href"])

        if parsed is None:
            continue

        rel_values = link.get("rel", [])

        if isinstance(rel_values, str):
            rel_values = rel_values.split()

        rel_values = [value.lower() for value in rel_values]

        is_category = (
            "/category/" in parsed.path.lower()
            or "category" in rel_values
        )

        is_tag = (
            "/tag/" in parsed.path.lower()
            or "tag" in rel_values
        )

        if is_tag and not is_category:
            tags.append(
                link.get_text(" ", strip=True)
            )

    for link in article.select(
        ".tags-links a, "
        ".post-tags a, "
        ".entry-tags a"
    ):
        _, parsed = safe_url_info(link.get("href"))

        if parsed and "/category/" not in parsed.path.lower():
            tags.append(
                link.get_text(" ", strip=True)
            )

    return unique_clean_values(tags)


# ------------------------------------------------------------
# 3.1.7 EXTRACT COMMENT COUNT
# ------------------------------------------------------------

def extract_comment_count(soup, article):
    """Extract the number of comments associated with an article."""

    for selector in [
        ".comments-link",
        ".comment-count",
        "a[href*='#comments']"
    ]:
        element = article.select_one(selector)

        if element:
            value = element.get_text(" ", strip=True)
            match = re.search(r"\d+", value)

            if match:
                return int(match.group())

            if re.search(
                r"keine Kommentare|no comments",
                value,
                flags=re.IGNORECASE
            ):
                return 0

    # Fallback: count visible comment elements.
    comments = soup.select(
        "#comments li.comment, "
        ".comment-list li.comment, "
        "ol.commentlist li.comment"
    )

    return len(comments)


# ------------------------------------------------------------
# 3.1.8 EXTRACT AND CLEAN ARTICLE TEXT
# ------------------------------------------------------------

def extract_clean_article_text(article, title):
    """
    Extract article body text while excluding navigation,
    publication metadata, sharing controls, and comments.
    """

    body = article.select_one(
        ".entry-content, "
        ".post-content, "
        ".article-content"
    )

    if body is None:
        body = article

    # Work on a copy to preserve the original article HTML.
    cleaned = BeautifulSoup(str(body), "lxml")

    remove_selectors = [
        "script", "style", "noscript", "iframe", "form",
        "nav", "header", "footer",
        "h1.entry-title", "h2.entry-title", ".entry-title",
        ".entry-meta", ".post-meta", ".postmetadata",
        ".post-meta-data", ".entry-footer", ".post-footer",
        ".byline", ".author", ".vcard", ".date", ".published",
        ".cat-links", ".tags-links", ".post-categories",
        ".entry-categories", ".post-tags", ".entry-tags",
        ".comments-link", ".comment-count",
        ".sharedaddy", ".sd-sharing-enabled",
        ".share-buttons", ".social-share",
        ".navigation", ".post-navigation",
        ".comments-area", "#comments", "#respond",
        ".leave-comment", ".comment-respond",
        ".related-posts", ".jp-relatedposts", ".yarpp-related"
    ]

    for selector in remove_selectors:
        for element in cleaned.select(selector):
            element.decompose()

    # Remove category and tag links outside the main text.
    for link in list(cleaned.select(
        "a[rel~='category'], "
        "a[rel~='tag'], "
        "a[href*='/category/'], "
        "a[href*='/tag/']"
    )):
        if link.parent is None:
            continue

        parent = link.parent
        parent_text = clean_spacing(
            parent.get_text(" ", strip=True)
        ).lower()

        metadata_terms = [
            "kategorie", "kategorien",
            "category", "categories",
            "schlagwort", "schlagwörter", "tags"
        ]

        if any(term in parent_text for term in metadata_terms):
            parent.decompose()
        else:
            link.decompose()

    text = clean_spacing(
        cleaned.get_text(separator="\n", strip=True)
    )

    # Remove a repeated article title at the beginning.
    if title and text.casefold().startswith(title.casefold()):
        text = text[len(title):].strip()

    # Remove publication metadata at the beginning of the text.
    text = re.sub(
        r"^\s*\d*\s*Veröffentlicht\s+von\s+.+?"
        r"\s+am\s+\d{1,2}\.\s+\w+\s+\d{4}\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove common category and tag footer lines.
    text = re.sub(
        r"\n\s*(?:Kategorien?|Categories?|"
        r"Schlagwörter?|Tags?)\s*:.*$",
        "",
        text,
        flags=re.IGNORECASE | re.MULTILINE
    )

    # Remove comment prompts at the end of the article.
    text = re.sub(
        r"\n\s*(?:Kommentar schreiben|"
        r"Hinterlasse einen Kommentar).*$",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    return clean_spacing(text)


# ------------------------------------------------------------
# 3.1.9 CALCULATE ARTICLE WORD COUNT
# ------------------------------------------------------------

def count_words(text):
    """Count words in the cleaned article body."""

    if not text:
        return 0

    words = re.findall(
        r"\b[0-9A-Za-zÀ-ÖØ-öø-ÿÄÖÜäöüß]+"
        r"(?:[-’'][0-9A-Za-zÀ-ÖØ-öø-ÿÄÖÜäöüß]+)*\b",
        text,
        flags=re.UNICODE
    )

    return len(words)


# ------------------------------------------------------------
# 3.1.10 DETECT TEMPORARY WEBSITE ANNOUNCEMENTS
# ------------------------------------------------------------

TEMPORARY_TITLES = {
    "vorübergehend statische seite",
    "temporarily static page"
}


def is_temporary_page(title, text):
    """Identify the temporary website announcement."""

    normalized_title = clean_spacing(title).casefold()
    normalized_text = clean_spacing(text).casefold()

    if normalized_title in TEMPORARY_TITLES:
        return True

    return (
        "wordpress wurde wiederholt angegriffen"
        in normalized_text
        and "blog" in normalized_text
        and len(normalized_text) < 1500
    )


# ------------------------------------------------------------
# 3.1.11 SCRAPE ONE COMPLETE BLOG POST
# ------------------------------------------------------------

def scrape_post(record):
    """
    Retrieve one article using its discovered static URL
    and return its content and metadata as a dictionary.
    """

    url = record["url"]
    soup = get_soup(url)
    article = find_article(soup)

    if article is None:
        raise ValueError(f"Main article not found: {url}")

    # Extract article metadata.
    title = extract_title(article)
    publication_date = extract_publication_date(article)
    author = extract_author(article)
    categories = extract_categories(article)
    tags = extract_tags(article)

    # Extract the cleaned article body.
    article_text = extract_clean_article_text(article, title)

    # Reject temporary announcements and incomplete articles.
    if is_temporary_page(title, article_text):
        raise ValueError(
            "Temporary website announcement returned"
        )

    if not title:
        raise ValueError("Missing article title")

    if not publication_date:
        raise ValueError("Missing publication date")

    if not article_text:
        raise ValueError("Empty article text")

    parsed_date = pd.to_datetime(
        publication_date,
        errors="coerce"
    )

    # Return article metadata in the established dataset format.
    return {
        "post_id": str(record["post_id"]),
        "url": url,
        "archive_month": record["archive_month"],
        "archive_page": record["archive_page"],
        "archive_page_url": record["archive_page_url"],
        "title": title,
        "publication_date": publication_date,
        "year": (
            int(parsed_date.year)
            if not pd.isna(parsed_date)
            else None
        ),
        "month": (
            int(parsed_date.month)
            if not pd.isna(parsed_date)
            else None
        ),
        "author": author,
        "categories": " | ".join(categories),
        "category_count": len(categories),
        "tags": " | ".join(tags),
        "tag_count": len(tags),
        "comments": extract_comment_count(soup, article),
        "word_count": count_words(article_text),
        "text": article_text
    }


print("SECTION 3 — PART 1 COMPLETED")
print("Article extraction functions are ready.")


SECTION 3 — PART 1 COMPLETED
Article extraction functions are ready.


#### 3.2	Full Article Scraping with Checkpoints

In [61]:
# ============================================================
# SECTION 3: ARTICLE EXTRACTION, SCRAPING, AND VALIDATION
# PART 2: FULL ARTICLE SCRAPING WITH CHECKPOINTS
# ============================================================
# Purpose:
# - Resume scraping from previously saved checkpoints.
# - Scrape posts not yet collected.
# - Save progress periodically to prevent data loss.
# - Record failed posts for subsequent retries.
# ============================================================


# ------------------------------------------------------------
# 3.2.1 CONFIGURE SCRAPING AND CHECKPOINTS
# ------------------------------------------------------------

if "scrape_post" not in globals():
    raise RuntimeError(
        "Run Section 3 Part 1 before starting full scraping."
    )

if "post_links_df" not in globals():
    raise RuntimeError(
        "Run Section 2 before starting full scraping."
    )

# Use the discovered post links from Section 2.
records_df = post_links_df.copy()

# Store checkpoints alongside the scraper notebook.
checkpoint_dir = PROJECT_ROOT / "notebooks"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_file = checkpoint_dir / "dhd_blog_checkpoint.csv"
checkpoint_errors_file = (
    checkpoint_dir / "dhd_blog_checkpoint_errors.csv"
)

CHECKPOINT_EVERY = 50
REQUEST_DELAY = 0.5

error_columns = [
    "post_id",
    "url",
    "archive_month",
    "archive_page",
    "error"
]


# ------------------------------------------------------------
# 3.2.2 PREPARE UNIQUE POST RECORDS
# ------------------------------------------------------------

records_df["post_id"] = records_df["post_id"].astype(str)

records_df = (
    records_df
    .drop_duplicates(subset=["post_id"])
    .reset_index(drop=True)
)

print("Total discovered posts:", len(records_df))


# ------------------------------------------------------------
# 3.2.3 LOAD EXISTING CHECKPOINTS
# ------------------------------------------------------------

if checkpoint_file.exists():
    df = pd.read_csv(
        checkpoint_file,
        dtype={"post_id": str},
        keep_default_na=False
    )
    print("Existing checkpoint loaded:", len(df))
else:
    df = pd.DataFrame()
    print("No checkpoint found. Starting fresh.")

if checkpoint_errors_file.exists():
    post_errors_df = pd.read_csv(
        checkpoint_errors_file,
        dtype={"post_id": str},
        keep_default_na=False
    )
else:
    post_errors_df = pd.DataFrame(columns=error_columns)

if not df.empty:
    df["post_id"] = df["post_id"].astype(str)
    df = (
        df.drop_duplicates(subset=["post_id"], keep="last")
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 3.2.4 IDENTIFY POSTS REQUIRING SCRAPING
# ------------------------------------------------------------

completed_ids = (
    set(df["post_id"])
    if not df.empty
    else set()
)

pending_records = records_df.loc[
    ~records_df["post_id"].isin(completed_ids)
].to_dict("records")

print("Already completed:", len(completed_ids))
print("Posts remaining:", len(pending_records))


# ------------------------------------------------------------
# 3.2.5 SCRAPE REMAINING POSTS
# ------------------------------------------------------------

new_rows = []
new_errors = []

for index, record in enumerate(
    tqdm(pending_records, desc="Scraping DHd Blog posts"),
    start=1
):
    try:
        result = scrape_post(record)

        if result is None:
            raise ValueError("scrape_post() returned None")

        new_rows.append(result)

    except Exception as error:
        new_errors.append({
            "post_id": str(record["post_id"]),
            "url": record["url"],
            "archive_month": record.get("archive_month", ""),
            "archive_page": record.get("archive_page", ""),
            "error": str(error)
        })

    # Save progress after every configured batch.
    if index % CHECKPOINT_EVERY == 0:

        if new_rows:
            df = pd.concat(
                [df, pd.DataFrame(new_rows)],
                ignore_index=True
            )
            new_rows = []

        if new_errors:
            post_errors_df = pd.concat(
                [post_errors_df, pd.DataFrame(new_errors)],
                ignore_index=True
            )
            new_errors = []

        if not df.empty:
            df["post_id"] = df["post_id"].astype(str)
            df = (
                df.drop_duplicates(
                    subset=["post_id"],
                    keep="last"
                )
                .reset_index(drop=True)
            )

        df.to_csv(
            checkpoint_file,
            index=False,
            encoding="utf-8-sig"
        )

        post_errors_df.to_csv(
            checkpoint_errors_file,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"Checkpoint saved: {len(df)} successful, "
            f"{len(post_errors_df)} recorded errors"
        )

    time.sleep(REQUEST_DELAY)


# ------------------------------------------------------------
# 3.2.6 MERGE FINAL RESULTS AND CLEAN ERROR RECORDS
# ------------------------------------------------------------

if new_rows:
    df = pd.concat(
        [df, pd.DataFrame(new_rows)],
        ignore_index=True
    )

if new_errors:
    post_errors_df = pd.concat(
        [post_errors_df, pd.DataFrame(new_errors)],
        ignore_index=True
    )

if not df.empty:
    df["post_id"] = df["post_id"].astype(str)
    df = (
        df.drop_duplicates(subset=["post_id"], keep="last")
        .reset_index(drop=True)
    )

# Remove errors for posts that have been successfully scraped.
if not post_errors_df.empty and not df.empty:
    successful_ids = set(df["post_id"])

    post_errors_df = (
        post_errors_df.loc[
            ~post_errors_df["post_id"].astype(str).isin(
                successful_ids
            )
        ]
        .drop_duplicates(subset=["post_id"], keep="last")
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 3.2.7 SAVE FINAL SCRAPING CHECKPOINTS
# ------------------------------------------------------------

df.to_csv(
    checkpoint_file,
    index=False,
    encoding="utf-8-sig"
)

post_errors_df.to_csv(
    checkpoint_errors_file,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 3.2.8 SUMMARIZE SCRAPING RESULTS
# ------------------------------------------------------------

discovered_ids = set(records_df["post_id"])
scraped_ids = set(df["post_id"])
missing_ids = discovered_ids - scraped_ids

print("\n========== FULL SCRAPING SUMMARY ==========")
print("Total discovered:", len(discovered_ids))
print("Successfully scraped:", len(scraped_ids))
print("Failed / missing:", len(missing_ids))
print("Recorded errors:", len(post_errors_df))
print("Duplicate post IDs:", df["post_id"].duplicated().sum())

print("\nCheckpoint:", checkpoint_file)
print("Error checkpoint:", checkpoint_errors_file)

if not df.empty:
    print("\nSample scraped posts:")
    display(
        df[
            [
                "post_id",
                "title",
                "publication_date",
                "author",
                "word_count"
            ]
        ].head(10)
    )

if not post_errors_df.empty:
    print("\nFailed posts:")
    display(post_errors_df.head(20))

print("\nSECTION 3 — PART 2 COMPLETED")
print("Published dataset files have not been overwritten.")


Total discovered posts: 2780
Existing checkpoint loaded: 2779
Already completed: 2779
Posts remaining: 1


Scraping DHd Blog posts: 100%|██████████| 1/1 [00:00<00:00,  1.61it/s]



========== FULL SCRAPING SUMMARY ==========
Total discovered: 2780
Successfully scraped: 2779
Failed / missing: 1
Recorded errors: 1
Duplicate post IDs: 0

Checkpoint: C:\Users\jiwani\Documents\dhd-blog-analysis\notebooks\dhd_blog_checkpoint.csv
Error checkpoint: C:\Users\jiwani\Documents\dhd-blog-analysis\notebooks\dhd_blog_checkpoint_errors.csv

Sample scraped posts:


,post_id,title,publication_date,author,word_count
0,23968,Aufeinander zugehen: Die Dokumentation des GND-Forums Humanities@NFDI vom 23./24. Juni 2026,2026-07-27,Anja Gerber,177
1,23987,"30. AWK: Data Ethics for Archival Science and Historical Research, Marburg 26./27.8.2026",2026-07-28,Robert Meier,205
2,23978,Professur für Bibliotheks- und Informationsmarketing W 2 | unbefristet | Vollzeit | TH Köln,2026-07-27,Mirjam Blümm,179
3,23972,Call for proposals: NFDI4Culture Art History Award (AHA),2026-07-27,Anja Gerber,482
4,23954,Bewerben bis 31.07.: Wiss. Mitarbeiter*in im Projekt „Heinrich Scholz und die Schule von Münster – Mathematische Logik und Grundlagenforschung“,2026-07-16,Christian Wachter,439
5,23966,Clustertreffen des N4O CC Data Capture and Creation zum Thema Culture Cloud am 8. September 2026,2026-07-27,Anja Gerber,135
6,23993,Call for Hosts DHd2029,2026-07-29,Vivien Wolter,495
7,23959,Bericht vom Hands-On Lab “Digital Footprint – Choose your own adventure” – Datentracking in der Wissenschaft spielerisch verstehen” bei der 114. BiblioCon in Berlin,2026-07-24,Ulrike Wuttke,697
8,23983,"Lecture: Tackling Data Gaps in Digital Humanities, 09.09.26, online",2026-07-28,Polina Solonets,280
9,23956,Bewerben bis 10.08.2026: Akad. Rätin/Akad. Rat auf Zeit „Digitale Geschichtswissenschaften“ (Univ. Münster),2026-07-21,Christian Wachter,42



Failed posts:


,post_id,url,archive_month,archive_page,error
0,24392,https://dhd-blog.org/index5f66.html?p=24392,Oktober 2026,1,Temporary website announcement returned



SECTION 3 — PART 2 COMPLETED
Published dataset files have not been overwritten.


#### 3.3	Error Recovery and Data Quality Validation

In [62]:
# ============================================================
# SECTION 3: ARTICLE EXTRACTION, SCRAPING, AND VALIDATION
# PART 3: ERROR RECOVERY AND DATA QUALITY VALIDATION
# ============================================================
# Purpose:
# - Retry posts that failed during initial scraping.
# - Check article completeness and extraction quality.
# - Identify repeated titles and texts without deleting them.
# - Verify scraping coverage and save recovery checkpoints.
# ============================================================


# ------------------------------------------------------------
# 3.3.1 RETRY FAILED POSTS
# ------------------------------------------------------------

MAX_RETRIES = 2

if not post_errors_df.empty:

    pending_ids = set(
        post_errors_df["post_id"].astype(str)
    )

    for attempt in range(1, MAX_RETRIES + 1):

        if not pending_ids:
            break

        print(
            f"\nRetry attempt {attempt}: "
            f"{len(pending_ids)} posts"
        )

        recovered_rows = []
        remaining_errors = []

        retry_records = records_df.loc[
            records_df["post_id"].isin(pending_ids)
        ].to_dict("records")

        for record in tqdm(
            retry_records,
            desc=f"Retry {attempt}"
        ):
            try:
                result = scrape_post(record)

                if result is None:
                    raise ValueError("scrape_post() returned None")

                recovered_rows.append(result)

            except Exception as error:
                remaining_errors.append({
                    "post_id": record["post_id"],
                    "url": record["url"],
                    "archive_month": record["archive_month"],
                    "archive_page": record["archive_page"],
                    "error": str(error)
                })

            time.sleep(1)

        if recovered_rows:
            df = pd.concat(
                [df, pd.DataFrame(recovered_rows)],
                ignore_index=True
            )

        post_errors_df = pd.DataFrame(
            remaining_errors,
            columns=error_columns
        )

        pending_ids = set(
            post_errors_df["post_id"].astype(str)
        )

        print("Still failed:", len(pending_ids))


# ------------------------------------------------------------
# 3.3.2 REMOVE DUPLICATE POST IDENTIFIERS
# ------------------------------------------------------------
# Deduplicate by post ID only, not by title or article text.

df["post_id"] = df["post_id"].astype(str)

df = (
    df.drop_duplicates(subset=["post_id"], keep="last")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3.3.3 CHECK REQUIRED ARTICLE FIELDS
# ------------------------------------------------------------

print("\n========== DATA QUALITY CHECKS ==========")

for column in [
    "title",
    "publication_date",
    "author",
    "text"
]:
    missing = (
        df[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"Missing {column}: {missing}")


# ------------------------------------------------------------
# 3.3.4 CHECK TITLE CONTAMINATION IN ARTICLE TEXT
# ------------------------------------------------------------
# Calculate this check without adding a column to the dataset.

text_starts_with_title = df.apply(
    lambda row: (
        bool(str(row["title"]).strip())
        and bool(str(row["text"]).strip())
        and str(row["text"]).strip().casefold().startswith(
            str(row["title"]).strip().casefold()
        )
    ),
    axis=1
)

print(
    "Texts starting with title:",
    text_starts_with_title.sum()
)


# ------------------------------------------------------------
# 3.3.5 IDENTIFY REPEATED ARTICLE TITLES
# ------------------------------------------------------------
# Repeated titles are informational, not automatic errors.

duplicate_titles = df.loc[
    df["title"].fillna("").duplicated(keep=False)
].sort_values(["title", "publication_date"])

print(
    "Posts sharing duplicate titles:",
    len(duplicate_titles)
)

if not duplicate_titles.empty:
    display(
        duplicate_titles[
            ["post_id", "title", "publication_date"]
        ].head(20)
    )


# ------------------------------------------------------------
# 3.3.6 IDENTIFY IDENTICAL ARTICLE TEXT
# ------------------------------------------------------------
# Distinct posts may legitimately publish identical content.

duplicate_texts = df.loc[
    df["text"].fillna("").duplicated(keep=False)
].sort_values("text")

print(
    "Posts sharing identical text:",
    len(duplicate_texts)
)

if not duplicate_texts.empty:
    display(
        duplicate_texts[
            ["post_id", "title", "word_count"]
        ].head(20)
    )


# ------------------------------------------------------------
# 3.3.7 IDENTIFY SHORT ARTICLES
# ------------------------------------------------------------
# Short posts are flagged for inspection, not removed.

short_posts = df.loc[
    pd.to_numeric(
        df["word_count"],
        errors="coerce"
    ) < 50,
    ["post_id", "title", "word_count"]
]

print(
    "Posts with fewer than 50 words:",
    len(short_posts)
)

display(short_posts.head(20))


# ------------------------------------------------------------
# 3.3.8 CHECK CATEGORY AND TAG COVERAGE
# ------------------------------------------------------------

print(
    "Posts with categories:",
    df["categories"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print(
    "Posts with tags:",
    df["tags"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)


# ------------------------------------------------------------
# 3.3.9 VERIFY ARCHIVE PAGINATION COVERAGE
# ------------------------------------------------------------

archive_pages = pd.to_numeric(
    df["archive_page"],
    errors="coerce"
)

print(
    "Posts from archive page 2 or later:",
    (archive_pages >= 2).sum()
)

print("Highest archive page:", archive_pages.max())


# ------------------------------------------------------------
# 3.3.10 VERIFY SCRAPING COMPLETENESS
# ------------------------------------------------------------

discovered_ids = set(
    records_df["post_id"].astype(str)
)

scraped_ids = set(
    df["post_id"].astype(str)
)

missing_ids = discovered_ids - scraped_ids

print("Discovered posts:", len(discovered_ids))
print("Successfully scraped:", len(scraped_ids))
print("Posts still missing:", len(missing_ids))
print(
    "Duplicate post IDs:",
    df["post_id"].duplicated().sum()
)


# ------------------------------------------------------------
# 3.3.11 CHECK TEMPORARY WEBSITE ANNOUNCEMENTS
# ------------------------------------------------------------

temporary_rows = df.loc[
    df.apply(
        lambda row: is_temporary_page(
            row["title"],
            row["text"]
        ),
        axis=1
    )
]

print(
    "Temporary announcement pages:",
    len(temporary_rows)
)


# ------------------------------------------------------------
# 3.3.12 SAVE RECOVERY CHECKPOINTS
# ------------------------------------------------------------

df.to_csv(
    checkpoint_file,
    index=False,
    encoding="utf-8-sig"
)

post_errors_df.to_csv(
    checkpoint_errors_file,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 3.3.13 DISPLAY FINAL DATA QUALITY SUMMARY
# ------------------------------------------------------------

print("\n========== FINAL SECTION 3 VALIDATION ==========")

print("Total discovered:", len(discovered_ids))
print("Successfully scraped:", len(df))
print("Still missing:", len(missing_ids))
print("Remaining errors:", len(post_errors_df))
print("Duplicate post IDs:", df["post_id"].duplicated().sum())

print("\nSample records:")

display(
    df[
        [
            "post_id",
            "title",
            "publication_date",
            "author",
            "categories",
            "tags",
            "word_count"
        ]
    ].head(10)
)

if not post_errors_df.empty:
    print("\nRemaining errors:")
    display(post_errors_df.head(20))

print("\nSECTION 3 — PART 3 COMPLETED")
print("Published dataset files have not been overwritten.")



Retry attempt 1: 1 posts


Retry 1: 100%|██████████| 1/1 [00:01<00:00,  1.11s/it]


Still failed: 1

Retry attempt 2: 1 posts


Retry 2: 100%|██████████| 1/1 [00:01<00:00,  1.10s/it]


Still failed: 1

========== DATA QUALITY CHECKS ==========
Missing title: 0
Missing publication_date: 0
Missing author: 0
Missing text: 0
Texts starting with title: 0
Posts sharing duplicate titles: 59


,post_id,title,publication_date
1607,9958,"DARIAH Code Sprint on Bibliographical Metadata, Berlin: 31.07.-02.08.2018",2018-05-30
1594,10159,"DARIAH Code Sprint on Bibliographical Metadata, Berlin: 31.07.-02.08.2018",2018-07-02
1824,7800,DH-SoftwareentwicklerIn an der Österreichischen Nationalbibliothek (Wien),2017-03-07
1771,8308,DH-SoftwareentwicklerIn an der Österreichischen Nationalbibliothek (Wien),2017-07-27
1112,14360,Datenkuratorin / Datenkurator (m/w/d) für die fotografischen Sammlungsbestände und digitale Langzeitarchivierung,2020-09-28
1106,14446,Datenkuratorin / Datenkurator (m/w/d) für die fotografischen Sammlungsbestände und digitale Langzeitarchivierung,2020-10-07
629,18750,Die Coordination Committees von Text+ wurden neu gewählt; das sind die Ergebnisse,2022-12-15
296,21800,Die Coordination Committees von Text+ wurden neu gewählt; das sind die Ergebnisse,2024-12-03
709,18073,Jetzt bewerben: Das CRETA-Coaching für Doktorand*innen!,2022-07-04
382,20997,Jetzt bewerben: Das CRETA-Coaching für Doktorand*innen!,2024-05-30


... (5 outputs left out)

,post_id,title,publication_date,author,categories,tags,word_count
0,23968,Aufeinander zugehen: Die Dokumentation des GND-Forums Humanities@NFDI vom 23./24. Juni 2026,2026-07-27,Anja Gerber,Allgemein | Forschungsdaten | Forschungsinfrastruktur | Reflektion,Dokumentation | Fishbowl-Diskussion | GND | NFDI | NFDI4Culture | NFDI4Memory | NFDI4Objects | Positionspapier | Text+ | Tools,177
1,23987,"30. AWK: Data Ethics for Archival Science and Historical Research, Marburg 26./27.8.2026",2026-07-28,Robert Meier,Allgemein,Datenethik,205
2,23978,Professur für Bibliotheks- und Informationsmarketing W 2 | unbefristet | Vollzeit | TH Köln,2026-07-27,Mirjam Blümm,Stellenanzeigen,Bibliotheks- und Informationswissenschaft | TH Köln,179
3,23972,Call for proposals: NFDI4Culture Art History Award (AHA),2026-07-27,Anja Gerber,Allgemein | Call,Arbeitskreis Digitale Kunstgeschichte | Datenpublikation | Digital Humanities | Digitale Kunstgeschichte | Forschungsdaten | Kunstgeschichte | NFDI4Culture,482
4,23954,Bewerben bis 31.07.: Wiss. Mitarbeiter*in im Projekt „Heinrich Scholz und die Schule von Münster – Mathematische Logik und Grundlagenforschung“,2026-07-16,Christian Wachter,Stellenanzeigen,digital philosophy | Digitale Edition | Digitale Editionen | Münster | Philosophie,439
5,23966,Clustertreffen des N4O CC Data Capture and Creation zum Thema Culture Cloud am 8. September 2026,2026-07-27,Anja Gerber,Forschungsdaten | Forschungsinfrastruktur | Veranstaltungen,Digitale Sammlungen | Kulturdaten | kulturelles Erbe | NFDI | NFDI4Objects | Sammlungen,135
6,23993,Call for Hosts DHd2029,2026-07-29,Vivien Wolter,Call | Konferenz,,495
7,23959,Bericht vom Hands-On Lab “Digital Footprint – Choose your own adventure” – Datentracking in der Wissenschaft spielerisch verstehen” bei der 114. BiblioCon in Berlin,2026-07-24,Ulrike Wuttke,Allgemein | Open Access | Porträt (Projekt) | Publizieren | Veranstaltungen,Datentracking | OER | Science Tracking | Workshop,697
8,23983,"Lecture: Tackling Data Gaps in Digital Humanities, 09.09.26, online",2026-07-28,Polina Solonets,Allgemein | Veranstaltungen | Vortrag,,280
9,23956,Bewerben bis 10.08.2026: Akad. Rätin/Akad. Rat auf Zeit „Digitale Geschichtswissenschaften“ (Univ. Münster),2026-07-21,Christian Wachter,Stellenanzeigen,Digital History | Stelle | Stellenangebot,42



Remaining errors:


,post_id,url,archive_month,archive_page,error
0,24392,https://dhd-blog.org/index5f66.html?p=24392,Oktober 2026,1,Temporary website announcement returned



SECTION 3 — PART 3 COMPLETED
Published dataset files have not been overwritten.


#### 3.4	Static URL Correction and Verification

In [63]:
# ============================================================
# SECTION 3: ARTICLE EXTRACTION, SCRAPING, AND VALIDATION
# PART 4: STATIC URL CORRECTION AND VERIFICATION
# ============================================================
# Purpose:
# - Match scraped posts to their discovered static URLs.
# - Replace outdated WordPress URLs with working static links.
# - Verify that all posts have a matching static URL.
# - Preserve all article metadata and text.
# ============================================================


# ------------------------------------------------------------
# 3.4.1 PREPARE POST IDENTIFIERS
# ------------------------------------------------------------

df["post_id"] = df["post_id"].astype(str)
post_links_df["post_id"] = post_links_df["post_id"].astype(str)


# ------------------------------------------------------------
# 3.4.2 BUILD STATIC URL MAPPING
# ------------------------------------------------------------

static_links = (
    post_links_df[["post_id", "url"]]
    .drop_duplicates(subset=["post_id"], keep="first")
)

url_mapping = static_links.set_index("post_id")["url"]


# ------------------------------------------------------------
# 3.4.3 UPDATE DATASET URLS
# ------------------------------------------------------------

# Preserve previous URLs for comparison.
old_urls = df["url"].copy()

# Replace URLs using the links discovered in Section 2.
df["url"] = (
    df["post_id"]
    .map(url_mapping)
    .fillna(df["url"])
)


# ------------------------------------------------------------
# 3.4.4 VALIDATE STATIC URL COVERAGE
# ------------------------------------------------------------

changed_urls = (old_urls != df["url"]).sum()

missing_links = (
    ~df["post_id"].isin(url_mapping.index)
).sum()

static_url_count = df["url"].str.contains(
    r"/index[^/]*\.html\?p=",
    regex=True,
    na=False
).sum()

print("\n========== STATIC URL VALIDATION ==========")
print("Total posts:", len(df))
print("URLs changed:", changed_urls)
print("Posts without discovered links:", missing_links)
print("Posts using static HTML URLs:", static_url_count)


# ------------------------------------------------------------
# 3.4.5 VERIFY URL CONSISTENCY
# ------------------------------------------------------------

assert missing_links == 0, (
    "Some posts have no discovered URL!"
)

assert static_url_count == len(df), (
    "Some posts do not have static HTML URLs!"
)

# Verify that each URL contains the correct post ID.
url_post_ids = df["url"].map(get_post_id)

assert url_post_ids.eq(df["post_id"]).all(), (
    "Some URLs do not match their post IDs!"
)


# ------------------------------------------------------------
# 3.4.6 PREVIEW CORRECTED ARTICLE URLS
# ------------------------------------------------------------

print("\nSample corrected post URLs:")

display(
    df.loc[
        df["post_id"].isin(["9958", "10159"]),
        ["post_id", "title", "url"]
    ]
)

print("\nSECTION 3 — PART 4 COMPLETED")
print("All dataset URLs have been validated.")
print("Run Section 4 to export the updated dataset.")



========== STATIC URL VALIDATION ==========
Total posts: 2779
URLs changed: 2698
Posts without discovered links: 0
Posts using static HTML URLs: 2779

Sample corrected post URLs:


,post_id,title,url
1594,10159,"DARIAH Code Sprint on Bibliographical Metadata, Berlin: 31.07.-02.08.2018",https://dhd-blog.org/indexb5de-2.html?p=10159
1607,9958,"DARIAH Code Sprint on Bibliographical Metadata, Berlin: 31.07.-02.08.2018",https://dhd-blog.org/indexebbb.html?p=9958



SECTION 3 — PART 4 COMPLETED
All dataset URLs have been validated.
Run Section 4 to export the updated dataset.


## 4. Final Dataset Validation and Export


In [64]:
# ============================================================
# SECTION 4: FINAL DATASET VALIDATION AND EXPORT
# ============================================================
# Purpose:
# - Validate the final scraped dataset before publication.
# - Ensure previously collected post IDs are preserved.
# - Export the dataset in CSV and Excel formats.
# - Save discovered post links and scraping errors.
# - Confirm output locations and final dataset statistics.
#
# Requires Sections 1–3 to be completed successfully.
# ============================================================


# ------------------------------------------------------------
# 4.1 DEFINE OUTPUT FILE PATHS
# ------------------------------------------------------------
# Use the project data directory configured in Section 1.

dataset_csv_path = DATA_DIR / "dhd_blog_dataset.csv"
dataset_xlsx_path = DATA_DIR / "dhd_blog_dataset.xlsx"
post_errors_path = DATA_DIR / "dhd_blog_post_errors.csv"
post_links_path = DATA_DIR / "dhd_blog_all_post_links.csv"


# ------------------------------------------------------------
# 4.2 VALIDATE FINAL DATASET
# ------------------------------------------------------------
# Check dataset completeness, unique post identifiers,
# column structure, and remaining scraping errors.

print("\n========== FINAL DATASET VALIDATION ==========")

print("Total scraped posts:", len(df))
print("Unique post IDs:", df["post_id"].astype(str).nunique())
print(
    "Duplicate post IDs:",
    df["post_id"].astype(str).duplicated().sum()
)
print("Total columns:", len(df.columns))
print("Remaining errors:", len(post_errors_df))
print("Total discovered links:", len(post_links_df))

# Prevent exporting an empty or incorrectly structured dataset.
assert not df.empty, "Dataset is empty!"

assert df["post_id"].astype(str).is_unique, (
    "Duplicate post IDs detected!"
)

assert len(df.columns) == 18, (
    "Unexpected dataset structure: expected 18 columns!"
)

# Verify that every scraped post appears in the discovered links.
discovered_ids = set(post_links_df["post_id"].astype(str))
current_ids = set(df["post_id"].astype(str))

assert current_ids.issubset(discovered_ids), (
    "Some scraped posts are missing from the discovered links!"
)


# ------------------------------------------------------------
# 4.3 COMPARE WITH PREVIOUSLY EXPORTED DATASET
# ------------------------------------------------------------
# Ensure that updating the dataset does not accidentally
# remove previously published posts.

if dataset_csv_path.exists():

    previous_df = pd.read_csv(
        dataset_csv_path,
        dtype={"post_id": str}
    )

    previous_ids = set(
        previous_df["post_id"].dropna()
    )

    missing_previous_ids = previous_ids - current_ids
    newly_added_ids = current_ids - previous_ids

    print("\n========== DATASET COMPARISON ==========")

    print("Previously exported posts:", len(previous_ids))
    print("Current scraped posts:", len(current_ids))
    print("Previously exported IDs missing:", len(missing_previous_ids))
    print("New post IDs:", len(newly_added_ids))

    assert not missing_previous_ids, (
        f"{len(missing_previous_ids)} previously exported "
        "post IDs are missing!"
    )

else:
    print("\nNo previously exported dataset found.")
    print("Existing dataset comparison skipped.")

print("\nDataset validation passed.")


# ------------------------------------------------------------
# 4.4 EXPORT FINAL DATASET AND SCRAPING RECORDS
# ------------------------------------------------------------
# Export the complete article dataset and supporting files.
# Existing files with the same names will be overwritten.

# Save the complete dataset as CSV.
df.to_csv(
    dataset_csv_path,
    index=False,
    encoding="utf-8-sig"
)

# Save the dataset as an Excel workbook.
# Excel cells have a maximum length of 32,767 characters.
# The CSV remains the primary full-text dataset.
df.to_excel(
    dataset_xlsx_path,
    index=False
)

# Save posts that could not be scraped successfully.
post_errors_df.to_csv(
    post_errors_path,
    index=False,
    encoding="utf-8-sig"
)

# Save all post links discovered from the blog archives.
post_links_df.to_csv(
    post_links_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 4.5 VERIFY EXPORTED FILES
# ------------------------------------------------------------
# Confirm that all four files were created successfully.

output_files = [
    dataset_csv_path,
    dataset_xlsx_path,
    post_errors_path,
    post_links_path
]

for file_path in output_files:
    assert file_path.exists(), (
        f"Output file was not created: {file_path}"
    )

print("\n========== EXPORTED FILES ==========")

print(f"Dataset CSV: {dataset_csv_path.resolve()}")
print(f"Dataset Excel: {dataset_xlsx_path.resolve()}")
print(f"Post errors: {post_errors_path.resolve()}")
print(f"Post links: {post_links_path.resolve()}")


# ------------------------------------------------------------
# 4.6 DISPLAY FINAL EXPORT SUMMARY
# ------------------------------------------------------------

print("\n========== FINAL EXPORT SUMMARY ==========")

print("Final dataset rows:", len(df))
print("Final dataset columns:", len(df.columns))
print("Unique post IDs:", df["post_id"].nunique())
print("Discovered post links:", len(post_links_df))
print("Remaining failed posts:", len(post_errors_df))
print("Files exported:", len(output_files))

print("\nSECTION 4 COMPLETED")
print("All four output files have been updated locally.")
print("GitHub has NOT been updated.")



========== FINAL DATASET VALIDATION ==========
Total scraped posts: 2779
Unique post IDs: 2779
Duplicate post IDs: 0
Total columns: 18
Remaining errors: 1
Total discovered links: 2780

========== DATASET COMPARISON ==========
Previously exported posts: 2779
Current scraped posts: 2779
Previously exported IDs missing: 0
New post IDs: 0

Dataset validation passed.


C:\Users\jiwani\AppData\Local\Temp\ipykernel_2856\1139277992.py:119: UserWarning: Cell contents too long (36590), truncated to 32767 characters
  df.to_excel(
C:\Users\jiwani\AppData\Local\Temp\ipykernel_2856\1139277992.py:119: UserWarning: Cell contents too long (48875), truncated to 32767 characters
  df.to_excel(
C:\Users\jiwani\AppData\Local\Temp\ipykernel_2856\1139277992.py:119: UserWarning: Cell contents too long (34541), truncated to 32767 characters
  df.to_excel(
C:\Users\jiwani\AppData\Local\Temp\ipykernel_2856\1139277992.py:119: UserWarning: Cell contents too long (32913), truncated to 32767 characters
  df.to_excel(



========== EXPORTED FILES ==========
Dataset CSV: C:\Users\jiwani\Documents\dhd-blog-analysis\data\dhd_blog_dataset.csv
Dataset Excel: C:\Users\jiwani\Documents\dhd-blog-analysis\data\dhd_blog_dataset.xlsx
Post errors: C:\Users\jiwani\Documents\dhd-blog-analysis\data\dhd_blog_post_errors.csv
Post links: C:\Users\jiwani\Documents\dhd-blog-analysis\data\dhd_blog_all_post_links.csv

========== FINAL EXPORT SUMMARY ==========
Final dataset rows: 2779
Final dataset columns: 18
Unique post IDs: 2779
Discovered post links: 2780
Remaining failed posts: 1
Files exported: 4

SECTION 4 COMPLETED
All four output files have been updated locally.
GitHub has NOT been updated.
